# Git Agent — user perspective

**What it does.** Takes Builder output for a finished work item and creates the feature branch + GitHub PR targeting the EPIC branch. Also handles `REBASE_STACK` after a sibling task PR merges.

**Entry point.** `run_git_agent(input: GitInput) -> GitOutput`.

**Hard rules.**
- Branch: `feature/LIN-{id}-{slug}` (GITA-01).
- PR title: `[LIN-{id}] {desc}` (GITA-03).
- PR base: the EPIC branch — never `main` directly (D-07).
- `git push --force-with-lease` only — no bare `--force` (Pitfall 3).
- `gh pr list --limit 100` to dodge pagination truncation (Pitfall 4).
- No merge tools, no Edit/Write, no Linear MCP (GITA-05).

**Cost.** Live runs create real branches and PRs in your repo on the current GitHub remote. Heavily gated.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Construct the input

`GitInput.implementation_output` is the serialized `BuilderOutput.model_dump()` — pass the dict directly. `epic_id` determines the PR base: `epic/LIN-{epic_id}`.

In [ ]:
from hsb.contracts.git import GitInput

# Stub of a BuilderOutput.model_dump() — in real use this is whatever Builder returned.
fake_builder_output = {
    "work_item_id": "LIN-456",
    "implementation_status": "completed",
    "summary": "Added docstring to scratch.py:greet().",
    "files_changed": [{"path": "scratch.py", "change_summary": "added docstring"}],
    "validation": {
        "build": "not_run",
        "tests": "passed",
        "lint": "passed",
        "typecheck": "not_run",
    },
    "implementation_notes": {
        "decisions": [],
        "assumptions": [],
        "risks": [],
        "qa_notes": [],
    },
}

example_input = GitInput(
    work_item_id="LIN-456",
    implementation_output=fake_builder_output,
    epic_id="123",
    dependencies=[],
)
print(example_input.model_dump_json(indent=2))

## Live invocation (heavily gated)

A live run will create a real branch and open a real PR. Gate with `HSB_NOTEBOOK_RUN_LIVE=1` only when you have a scratch repo + scratch EPIC branch ready. The cell remains a guarded scaffold by default.

In [ ]:
from _helpers import assert_g1_safe, gated, live_mode

if not live_mode():
    print(gated("Git Agent live run — opens a real PR"))
else:
    assert_g1_safe()
    from hsb.agents.git_agent import run_git_agent

    out = run_git_agent(example_input)
    print("branch:", out.branch)
    print("PR:   ", out.pull_request.url)
    print("title:", out.pull_request.title)